### Part 1: Code Extraction from the Lab Document

In [1]:
import torch

# Check if CUDA is available
device = torch.device('cpu')
if torch.cuda.is_available():
    device = torch.device('cuda')

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cuda:0


In [2]:
import string
import unicodedata

# We can use "_" to represent an out-of-vocabulary character
allowed_characters = string.ascii_letters + " .,;'"
n_letters = len(allowed_characters)

# Turn a Unicode string to plain ASCII
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in allowed_characters
    )

In [3]:
# Find letter index from all_letters, e.g. "a" = 0
def letterToIndex(letter):
    # Return our out-of-vocabulary character if we encounter a letter unknown to our model
    if letter not in allowed_characters:
        return allowed_characters.find("_")
    else:
        return allowed_characters.find(letter)

# Turn a line into a <line_length x 1 x n_letters> tensor
def lineToTensor(line):
    tensor = torch.zeros(len(line), 1, n_letters)
    for li, letter in enumerate(line):
        tensor[li][0][letterToIndex(letter)] = 1
    return tensor

In [4]:
from io import open
import glob
import os
import time
from torch.utils.data import Dataset

class NamesDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        self.load_time = time.localtime()
        labels_set = set()
        self.data = []
        self.data_tensors = []
        self.labels = []
        self.labels_tensors = []

        # Read all the .txt files in the specified directory
        text_files = glob.glob(os.path.join(data_dir, "*.txt"))
        for filename in text_files:
            label = os.path.splitext(os.path.basename(filename))[0]
            labels_set.add(label)
            lines = open(filename, encoding="utf-8").read().strip().split('\n')
            for name in lines:
                self.data.append(name)
                self.data_tensors.append(lineToTensor(name))
                self.labels.append(label)

        # Cache the tensor representation of the labels
        self.labels_uniq = list(labels_set)
        for idx in range(len(self.labels)):
            temp_tensor = torch.tensor([self.labels_uniq.index(self.labels[idx])], dtype=torch.long)
            self.labels_tensors.append(temp_tensor)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data_item = self.data[idx]
        data_label = self.labels[idx]
        data_tensor = self.data_tensors[idx]
        label_tensor = self.labels_tensors[idx]
        return label_tensor, data_tensor, data_label, data_item

In [5]:
import torch.nn as nn
import torch.nn.functional as F

class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(CharRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size)
        self.h2o = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, line_tensor):
        rnn_out, hidden = self.rnn(line_tensor)
        output = self.h2o(hidden[0])
        output = self.softmax(output)
        return output

In [6]:
def label_from_output(output, output_labels):
    top_n, top_i = output.topk(1)
    label_i = top_i[0].item()
    return output_labels[label_i], label_i

In [7]:
import random
import numpy as np

def train(rnn, training_data, n_epoch=10, n_batch_size=64, report_every=50, learning_rate=0.2, criterion=nn.NLLLoss()):
    all_losses = []
    rnn.train()
    optimizer = torch.optim.SGD(rnn.parameters(), lr=learning_rate)
    start = time.time()
    print(f"training on data set with n = {len(training_data)}")

    for iter in range(1, n_epoch + 1):
        rnn.zero_grad()
        batches = list(range(len(training_data)))
        random.shuffle(batches)
        batches = np.array_split(batches, max(1, len(batches) // n_batch_size))

        for idx, batch in enumerate(batches):
            batch_loss = 0
            for i in batch:
                (label_tensor, text_tensor, label, text) = training_data[i]
                output = rnn.forward(text_tensor)
                loss = criterion(output, label_tensor)
                batch_loss += loss

            batch_loss.backward()
            nn.utils.clip_grad_norm_(rnn.parameters(), 3)
            optimizer.step()
            optimizer.zero_grad()

            current_loss = batch_loss.item() / len(batch)
            all_losses.append(current_loss)

            if iter % report_every == 0:
                print(f" iter {iter} ({iter/n_epoch:.0%}): \t average batch loss {all_losses[-1]}")

    return all_losses

In [8]:
def evaluate(rnn, testing_data, classes):
    confusion = torch.zeros(len(classes), len(classes))
    rnn.eval()
    with torch.no_grad():
        for i in range(len(testing_data)):
            (label_tensor, text_tensor, label, text) = testing_data[i]
            output = rnn(text_tensor)
            guess, guess_i = label_from_output(output, classes)
            label_i = classes.index(label)
            confusion[label_i][guess_i] += 1

    # Normalize by dividing every row by its sum
    for i in range(len(classes)):
        denom = confusion[i].sum()
        if denom > 0:
            confusion[i] = confusion[i] / denom

    return confusion

### Part 2: Completing the Lab Task

The end of your document tasks you with adjusting hyperparameters, trying `nn.LSTM` and `nn.GRU`, and modifying the size of the layers (adding hidden nodes and linear layers).

Here is the code to fulfill those lab requirements.

In [9]:
class CharAdvancedRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, model_type="LSTM"):
        super(CharAdvancedRNN, self).__init__()
        self.model_type = model_type

        # 1. Using nn.LSTM or nn.GRU instead of standard RNN
        if model_type == "LSTM":
            self.rnn = nn.LSTM(input_size, hidden_size)
        elif model_type == "GRU":
            self.rnn = nn.GRU(input_size, hidden_size)

        # 2. Adding an extra linear layer and modifying layer sizes
        self.fc1 = nn.Linear(hidden_size, hidden_size // 2)
        self.fc2 = nn.Linear(hidden_size // 2, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, line_tensor):
        if self.model_type == "LSTM":
            rnn_out, (hidden, cell) = self.rnn(line_tensor)
        else: # For GRU
            rnn_out, hidden = self.rnn(line_tensor)

        # Passing output through the new extra layer
        x = F.relu(self.fc1(hidden[0]))
        output = self.fc2(x)
        output = self.softmax(output)
        return output

# 3. Adjusting Hyperparameters:
# Increased hidden nodes (128 to 256)
# Increased epochs (27 to 35)
# Decreased batch size (64 to 32)
# Lowered learning rate (0.15 to 0.05)
advanced_model = CharAdvancedRNN(n_letters, hidden_size=256, output_size=18, model_type="LSTM")

# You would train the new model like this:
# all_losses_advanced = train(advanced_model, train_set, n_epoch=35, n_batch_size=32, learning_rate=0.05)